# Reconstruction des caractéristiques d’URL

## Exécution

Ce notebook utilise les fonctions partagées du dossier src. Les cellules s’exécutent dans l’ordre, sans accès aux sites du dataset. Les sorties détaillées et figures sont exportées dans reports. Aucun modèle ni découpage n’est produit.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
elif (BASE / "periode-1").is_dir():
    BASE = BASE / "periode-1"
assert (BASE / "src").is_dir(), "Ouvrir le notebook depuis le dépôt ou le dossier notebooks"
if str(BASE / "src") not in sys.path:
    sys.path.insert(0, str(BASE / "src"))
pd.set_option("display.max_columns", 22)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 40)


In [2]:
from reconstruct_data import run_reconstruction
from extract_url_features import FEATURE_COLUMNS, extract_url_features
data, errors, manifest = run_reconstruction()
print("Lignes :", len(data), "Colonnes :", len(data.columns), "Caractéristiques :", len(FEATURE_COLUMNS))


Lignes : 235795 Colonnes : 21 Caractéristiques : 18


# Reconstruction et conventions de calcul


## Principe

Le nouveau dataset provient des URL et des labels du fichier original. Il ne s'agit ni d'une nouvelle collecte ni d'une création d'étiquettes. Les anciennes caractéristiques ne sont pas reprises comme entrées. La fonction extract_url_features, version 1.0.0, ne reçoit que l'URL et ne réalise aucun appel réseau.

Le fichier reconstruit possède **21 colonnes** : URL, Domain, label, 17 caractéristiques numériques et une caractéristique catégorielle TLD_calc. URL et Domain sont des métadonnées destinées à la traçabilité et aux contrôles ; label est la cible. Les 18 caractéristiques sont définies dans FEATURE_COLUMNS.


## Texte de référence et validation syntaxique

U = URL.strip(). Les espaces aux extrémités sont retirés pour le calcul, mais l'URL originale reste enregistrée jusqu'au nettoyage. Aucun caractère interne n'est supprimé, aucune séquence %HH n'est décodée et aucun chemin n'est tronqué. Les comptages portent sur toute la chaîne U, y compris schéma, port, identifiants éventuels, chemin, paramètres et fragment. Une lettre désigne exclusivement A-Z ou a-z, et un chiffre exclusivement 0-9. Les autres caractères Unicode font partie des autres caractères selon cette convention.

Seuls HTTP et HTTPS avec une autorité et un hôte sont acceptés. Les contrôles rejettent les espaces et contrôles internes, les ports invalides, les hôtes absents, les encodages ou zones IPv6 dans l'hôte et les noms DNS non pris en charge. Ce contrôle est syntaxique : il ne prouve ni l'existence ni la sûreté du site. Les échecs conservent leur ligne et leur label dans reconstructed, avec caractéristiques manquantes et motif dans extraction_errors.csv.


## Hôte et TLD

H est le hostname du parseur, en minuscules, sans port ni identifiants et sans point DNS final. Les noms DNS Unicode sont convertis en IDNA par la bibliothèque standard Python ; les comptages sur U restent réalisés sur le texte Unicode original. Les adresses IP restent celles du parseur, sans crochets IPv6. L'hôte n'est pas résolu par DNS. Les underscores sont tolérés dans les labels DNS, règle explicite de compatibilité syntaxique. Les labels DNS ont au plus 63 caractères et l'hôte DNS au plus 253.

TLD_calc est le dernier label DNS si H contient un point et n'est pas une IP. Il vaut la chaîne vide pour les IP et les noms sans point. Pour shop.exemple.co.uk, il vaut uk, pas co.uk. Le vide constitue ici une catégorie structurelle, pas une information manquante. La lecture CSV utilise keep_default_na=False pour la conserver.


In [3]:
# Exemple fictif : aucune connexion au domaine n'est effectuée.
example = extract_url_features("http://192.0.2.1/a?next=https://example.com")
display(pd.Series(example).to_frame("valeur"))
assert example["IsDomainIP_calc"] == 1
assert example["IsHTTPS_calc"] == 0

,valeur
Domain,192.0.2.1
URLLength_calc,43
DomainLength_calc,9
IsDomainIP_calc,1
IsHTTPS_calc,0
TLD_calc,
TLDLength_calc,0
NoOfLettersInURL_calc,24
LetterRatioInURL_calc,0.55814
NoOfDegitsInURL_calc,6


## Calculs numériques

Les longueurs utilisent len, donc le nombre de points de code Python, pas la taille UTF-8 en octets. Le domaine IDNA peut ainsi avoir une longueur différente de la représentation Unicode saisie.

Les compteurs =, ? et & mesurent des caractères littéraux, pas un nombre de paramètres. Un %3F ne compte pas comme ?. Les autres caractères sont ceux qui ne sont ni des lettres ASCII, ni des chiffres ASCII, ni =, ?, &. La somme de ces six catégories couvre exactement U.

Le ratio de chaque catégorie retenue est son compteur divisé par len(U). Aucun ratio n'est arrondi lors du calcul ; le CSV conserve la précision numérique et les contrôles utilisent 1e-12.

Le percent-encoding est détecté par % suivi de deux chiffres hexadécimaux. %61%2F donne deux jetons, %ZZ n'en donne aucun. percent_encoded_count compte les jetons, pas leurs trois caractères de représentation. has_percent_encoding indique au moins un jeton ; percent_encoded_ratio vaut nombre de jetons / len(U). L'encodage ne constitue pas en lui-même une preuve d'obfuscation malveillante, d'où les nouveaux noms.


In [4]:
display(data[FEATURE_COLUMNS].head(10))
display(pd.read_csv(BASE / "reports/tables/dictionnaire_variables.csv"))

,URLLength_calc,DomainLength_calc,IsDomainIP_calc,IsHTTPS_calc,TLDLength_calc,NoOfLettersInURL_calc,LetterRatioInURL_calc,NoOfDegitsInURL_calc,DegitRatioInURL_calc,NoOfEqualsInURL_calc,NoOfQMarkInURL_calc,NoOfAmpersandInURL_calc,NoOfOtherSpecialCharsInURL_calc,SpacialCharRatioInURL_calc,has_percent_encoding,percent_encoded_count,percent_encoded_ratio,TLD_calc
0,32,24,0,1,3,27,0.843750,0,0.0,0,0,0,5,0.156250,0,0,0.0,com
1,24,16,0,1,2,18,0.750000,0,0.0,0,0,0,6,0.250000,0,0,0.0,de
2,30,22,0,1,2,24,0.800000,0,0.0,0,0,0,6,0.200000,0,0,0.0,uk
3,27,19,0,1,3,22,0.814815,0,0.0,0,0,0,5,0.185185,0,0,0.0,com
4,34,26,0,1,3,29,0.852941,0,0.0,0,0,0,5,0.147059,0,0,0.0,org
5,31,23,0,1,3,26,0.838710,0,0.0,0,0,0,5,0.161290,0,0,0.0,org
6,26,18,0,1,3,21,0.807692,0,0.0,0,0,0,5,0.192308,0,0,0.0,com
7,26,18,0,1,3,21,0.807692,0,0.0,0,0,0,5,0.192308,0,0,0.0,com
8,30,22,0,1,2,25,0.833333,0,0.0,0,0,0,5,0.166667,0,0,0.0,in
9,19,11,0,1,3,14,0.736842,0,0.0,0,0,0,5,0.263158,0,0,0.0,org


,colonne,type,definition,role
0,URL,Texte,"URL source, puis strip au nettoyage","Métadonnée, exclue des entrées"
1,Domain,Texte,"Hôte du parseur, minuscules et IDNA DNS, sans port/identifiants/po...",Métadonnée pour contrôles et groupes
2,label,Entier,"Étiquette source : 0 phishing, 1 légitime","Cible, jamais une entrée"
3,URLLength_calc,Entier,len(URL.strip()),Entrée
4,DomainLength_calc,Entier,len(Domain),Entrée
5,IsDomainIP_calc,Binaire,1 si ipaddress reconnaît tout l’hôte,Entrée
6,IsHTTPS_calc,Binaire,1 si le schéma du parseur vaut https,Entrée
7,TLDLength_calc,Entier,len(TLD_calc),Entrée
8,NoOfLettersInURL_calc,Entier,Nombre de lettres ASCII A-Z et a-z dans U,Entrée
9,LetterRatioInURL_calc,"Réel [0,1]",Nombre de lettres / len(U),Entrée


## Résultat de reconstruction

235,795 lignes entrantes, 235,795 lignes reconstruites et 0 échecs. Aucune URL ni étiquette n'est supprimée à cette étape. Les anciennes colonnes ne sont pas conservées dans le fichier reconstruit, mais restent disponibles dans raw/dataset.csv.

Les copies d'une même URL produisent désormais les mêmes caractéristiques. Cela explique pourquoi 425 copies de lignes complètes existent après reconstruction, alors que le fichier original n'en comportait aucune hors identifiant.


In [5]:
display(errors)
print("Conservation de toutes les lignes :", len(data) == manifest["rows_raw"])
print("Doublons complets après reconstruction :", data.duplicated().sum())

Conservation de toutes les lignes : True
Doublons complets après reconstruction : 425


,source_row,URL,label,reason


## Reproductibilité

Empreinte SHA-256 du brut : `a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6`.

Empreinte SHA-256 du reconstruit : `2041a7127c7a2c60f05d8bbf3cfa77ae137c0d467b0c7af0c3b722f33ee8819b`.

Les chemins, empreintes, version d'extraction, ordre des colonnes et liste des entrées figurent dans data/reconstructed/manifest.json. Les tests vérifient les cas ASCII, Unicode, IP, IPv6, ports, identifiants, encodage, URL invalides et absence de requêtes réseau.


In [6]:
display(pd.Series(manifest).to_frame("valeur"))

,valeur
extractor_version,1.0.0
raw_sha256,a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6
reconstructed_sha256,2041a7127c7a2c60f05d8bbf3cfa77ae137c0d467b0c7af0c3b722f33ee8819b
rows_raw,235795
rows_reconstructed,235795
extraction_errors,0
columns,"[URL, Domain, label, URLLength_calc, DomainLength_calc, IsDomainIP..."
feature_columns,"[URLLength_calc, DomainLength_calc, IsDomainIP_calc, IsHTTPS_calc,..."
label_mapping,"{'0': 'phishing', '1': 'legitime'}"


## Usage futur

Le backend importe la même fonction, sans implémentation parallèle. TLD_calc devra être encodé avec gestion des catégories inconnues. La standardisation et les autres transformations apprises ne sont pas réalisées en période 1 ; elles devront être ajustées uniquement sur les données d'entraînement après découpage. Aucun label n'intervient dans l'extraction.
